In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import time
from birddog.core import (
    Archive,
    )
from birddog.wiki import get_all_pages, mw_read_page, canonicalize_title, get_recent_changes, PageTracker
from birddog.runtime import Runtime

2025-08-24 13:34:30,163 [INFO] Using Google Cloud translation API (credentials file:/Users/jbrandt/code/birddog/google-cloud-translate-key.json)
2025-08-24 13:34:30,169 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.


In [ ]:
titles = [ 
    "Архів:ДАПО/Р-9126/2",
    "Архів:ДАЖО/1/1",
    "Архів:ДАК/312/1",
    "Архів:ДАЖО/1/74",
    "Архів:Лука_Мала",
    "Архів:ЦДІАК/1/1",
    "Архів:ДАХО/Д",
    "Архів:ДАПО/Р/1–1000",
    "Архів:ЦДІАК/28",
    "Архів:ЦДІАК/28/1",
    "Архів:ДАЖО/1",
    "Архів:ДАК/Р-352",
    "Архів:Архівний відділ виконавчого комітету Кременчуцької міської ради/Р",
    "Архів:ДАЖО/Д",
    "Архів:ДАДнО/Р-6478/2", 
    "Архів:ДАЖО/752", 
    "Архів:ДАКрО/225/1/25", 
    "Архів:ДАКрО/225", 
    "Архів:ДАСО/Р", 
    "Архів:ДАХмО/К", 
    "Архів:ДАКрО/225/1/144а", 
    "Архів:ДАПО/978/1",
    "Архів:ДАПО/Р",
    "Архів:ДАХмО/Р-6193",
    "Архів:ДАКрО/П-5907/2Р",
    "Архів:ДАОО/Р-8085/1",
    "Архів:ДАКрО/225/1",
    "Архів:ДАПО/1072/1/1",
    "Архів:ДАПО/978/1/135",
    "Архів:ДАПО/978",
    "Архів:ДАКО/Р-5634/1/3092",
    ]

In [ ]:
runtime = Runtime()

In [ ]:
address = runtime.lookup_address("Архів:ДАПО/Р/9126/2")
print(address)

In [ ]:
runtime.lookup_by_address(*address)
print(address)

In [ ]:
runtime.lookup_by_address(*address).name

In [ ]:
for title in titles:
    address = runtime.lookup_address(title)
    page = runtime.lookup(*address)
    print(f"{title}: {address}, {page.report}")
    assert canonicalize_title(title) == canonicalize_title(page.title)

In [ ]:
def select_parent_archive(archive_root, fond_id, ti=manager._title_index):
    parent_archive = None
    known_child = False
    if not archive_root in ti._archives:
        raise ValueError(f"Unknown archive root: {archive_root}")
    for archive_address in ti._archives[archive_root]:
        archive = ti._lru.lookup(*archive_address)
        #print(archive.title)
        if fond_id.upper().startswith(archive.subarchive["uk"]):
            parent_archive = archive_address
            known_child = fond_id in archive.child_ids
            break
        elif archive_address[1] == "D" or len(ti._archives[archive_root]) == 1:
            parent_archive = archive_address
            known_child = fond_id in archive.child_ids
    return parent_archive, known_child

In [ ]:
def test_title(title, ti=manager._title_index):
    try:
        address = ti.lookup(title)
        return "known"
        #print("known:", title, address)
    except ValueError:
        title = canonicalize_title(title)
        title_split = title.split("/")
        if len(title_split) > 1:
            try:
                parent_archive, known_fond = select_parent_archive(*title_split[:2], ti)
                if not known_fond:
                    #print("need to adopt:", parent_archive, title)
                    return "adopt"
                else:
                    #print("something unexpected:", title)
                    return "unexpected"
            except ValueError:
                #print("unknown archive root:", title)
                return "unknown_archive"
        else:
            #print("unknown archive:", title)
            return "unknown_archive"

In [ ]:
test_title("Архів:ДАЧкО/5899/1")

In [5]:
updates = get_recent_changes()

2025-08-24 13:35:03,222 [INFO] fetch_url: 46 requests in last 60s → 0.77 req/s
2025-08-24 13:35:13,244 [INFO] fetch_url: 72 requests in last 60s → 1.20 req/s


In [6]:
len(updates)

11013

In [7]:
for k in list(updates.keys())[:10]:
    print (k, updates[k])

Архів:ДАХмО/18/1/407 {'timestamp': '2025,08,24,19:31', 'user': 'Alexkrakovsky'}
Архів:ДАХмО/18/1/406 {'timestamp': '2025,08,24,19:31', 'user': 'Alexkrakovsky'}
Архів:ДАХмО/18/1/405 {'timestamp': '2025,08,24,19:28', 'user': 'Alexkrakovsky'}
Архів:ДАХмО/18/1/404а {'timestamp': '2025,08,24,18:56', 'user': 'Alexkrakovsky'}
Архів:ДАХмО/18/1/404 {'timestamp': '2025,08,24,18:55', 'user': 'Alexkrakovsky'}
Архів:ДАХмО/18/1/403 {'timestamp': '2025,08,24,18:44', 'user': 'Alexkrakovsky'}
Архів:ДАХмО/18/1/402 {'timestamp': '2025,08,24,18:37', 'user': 'Alexkrakovsky'}
Архів:ДАКО/280/2/9 {'timestamp': '2025,08,24,17:15', 'user': 'Jbuket'}
Архів:ДАЖО/1/75/6 {'timestamp': '2025,08,24,16:54', 'user': '84.40.221.244'}
Архів:ДАЖО/1/75/5 {'timestamp': '2025,08,24,16:34', 'user': '84.40.221.244'}


In [9]:
min([v["timestamp"] for v in updates.values()])

'2025,07,25,19:57'

In [10]:
pgs = get_all_pages()

2025-08-24 13:36:25,110 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2025-08-24 13:36:35,114 [INFO] fetch_url: 45 requests in last 60s → 0.75 req/s
2025-08-24 13:36:45,233 [INFO] fetch_url: 90 requests in last 60s → 1.50 req/s
2025-08-24 13:36:55,401 [INFO] fetch_url: 135 requests in last 60s → 2.25 req/s
2025-08-24 13:37:05,630 [INFO] fetch_url: 181 requests in last 60s → 3.02 req/s
2025-08-24 13:37:05,631 [INFO] fetch_url: rate limit exceeded - sleeping...
2025-08-24 13:37:15,841 [INFO] fetch_url: 204 requests in last 60s → 3.40 req/s
2025-08-24 13:37:15,842 [INFO] fetch_url: rate limit exceeded - sleeping...
2025-08-24 13:37:25,991 [INFO] fetch_url: 223 requests in last 60s → 3.72 req/s
2025-08-24 13:37:25,992 [INFO] fetch_url: rate limit exceeded - sleeping...
2025-08-24 13:37:36,147 [INFO] fetch_url: 201 requests in last 60s → 3.35 req/s
2025-08-24 13:37:36,149 [INFO] fetch_url: rate limit exceeded - sleeping...
2025-08-24 13:37:46,165 [INFO] fetch_url: 178 requests in la

In [11]:
len(pgs)

152311

In [3]:
tracker = PageTracker()

In [4]:
updates = tracker.update()

2025-08-24 13:34:43,068 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2025-08-24 13:34:53,073 [INFO] fetch_url: 27 requests in last 60s → 0.45 req/s
2025-08-24 13:34:54,166 [INFO] TIMER: update.get_recent_changes took 11.513s
2025-08-24 13:34:54,229 [INFO] TIMER: _update_mod_dates.get_newer_updates took 0.053s
2025-08-24 13:34:57,237 [INFO] TIMER: _update_mod_dates.batch_page_exists took 3.007s
2025-08-24 13:34:57,247 [INFO] TIMER: _update_mod_dates.batch_store_updates took 0.008s


In [9]:
len(updates)

TypeError: object of type 'bool' has no len()

In [10]:
updates

False